# Part 10 — Calibration: Does a Stated Confidence Mean What It Says?

*Evals from First Principles, Part 10.*

A confidence is a promise: of the answers a model tags "80% sure," about 80% should be right. This notebook keeps that promise honest. Ten trivia answers each carry a stated confidence and a correct/wrong outcome. We bin them by confidence, build the reliability curve (mean confidence vs. accuracy per bin) by hand, then reduce the whole picture to two numbers: the Expected Calibration Error (ECE, the weighted average gap between confidence and accuracy) and the Brier score (mean squared error of confidence against outcome). We run it twice on the SAME confidences, once for an over-confident model and once for a well-calibrated one, so the metrics move while the confidences stay put.

Pure Python, offline, deterministic. No NumPy needed, no randomness, no network.

Companion script: `calibration.py`

In [ ]:
# --- The task ---------------------------------------------------------
# A trivia model answers 10 questions. For each it reports how sure it is
# (CONF, a probability in [0, 1]) and we later learn whether it was actually
# right (OUTCOME, 1 = correct, 0 = wrong). CONF[i] and OUTCOME[i] describe
# the same question. We keep the SAME confidences for both models below;
# only the outcomes differ, which is the whole point: calibration is about
# whether the outcomes track the confidences, not about the confidences alone.
CONF = [0.6, 0.6, 0.7, 0.8, 0.8, 0.9, 0.9, 0.9, 1.0, 1.0]

# Over-confident: it says 0.82 on average but is right only 40% of the time.
OUTCOME_OVER = [1, 0, 0, 1, 0, 1, 0, 0, 1, 0]

# Well-calibrated: same confidences, but the outcomes now track them.
OUTCOME_CAL = [1, 1, 0, 1, 1, 1, 1, 0, 1, 1]

# Reliability bins: three buckets that cover the confidences we see.
# Each bin is [lo, hi); the last one includes the right edge so 1.0 lands in it.
BINS = [(0.6, 0.8), (0.8, 1.0), (1.0, 1.0)]

print(f"{len(CONF)} predictions. confidences: {CONF}")

## Step 1 — Which bin does a confidence fall in?

We cannot compare "confidence" to "accuracy" one prediction at a time (accuracy only exists across a group). So the first step is grouping: sort every confidence into one of three half-open bins, `[0.6, 0.8)`, `[0.8, 1.0)`, and the degenerate top bin `[1.0]` for confidences that hit the ceiling exactly. `bin_index` is the by-hand version of `numpy.digitize`: walk the bin edges in order and return the first one that fits, with the top bin handled as a special case since its low and high edge are equal.

In [ ]:
def bin_index(conf):
    """Which bin does this confidence fall in? Half-open [lo, hi), last is closed."""
    for i, (lo, hi) in enumerate(BINS):
        if lo == hi:                      # the degenerate top bin, exactly 1.0
            if conf == lo:
                return i
        elif lo <= conf < hi:
            return i
    return len(BINS) - 1                  # anything at the top edge -> last bin


for c in [0.6, 0.75, 0.8, 0.95, 1.0]:
    print(f"  conf={c} -> bin {bin_index(c)} {BINS[bin_index(c)]}")

## Step 2 — Per-bin mean confidence and accuracy

With `bin_index` in hand, we can sweep all 10 predictions once and, for each bin, accumulate three running totals: how many predictions landed there (`n`), the sum of their confidences (to later average into `mean_conf`), and the sum of their outcomes (to later average into `accuracy`, since outcomes are 0/1). This is the by-hand version of `groupby(bin).agg(mean)` — one pass, three accumulators per bin, then a division at the end.

In [ ]:
def bin_stats(conf, outcome):
    """Group predictions into bins; return per-bin (n, mean_conf, accuracy)."""
    n = len(BINS)
    counts = [0] * n
    conf_sum = [0.0] * n
    correct = [0] * n
    for c, o in zip(conf, outcome):
        b = bin_index(c)
        counts[b] += 1
        conf_sum[b] += c
        correct[b] += o
    rows = []
    for i in range(n):
        if counts[i] == 0:
            rows.append((0, None, None))
        else:
            rows.append((counts[i], conf_sum[i] / counts[i], correct[i] / counts[i]))
    return rows


for count, mean_conf, acc in bin_stats(CONF, OUTCOME_OVER):
    print(f"  n={count}  mean_conf={mean_conf}  accuracy={acc}")

## Step 3 — Two ways to collapse the reliability curve into one number

The reliability curve (mean confidence vs. accuracy, per bin) is the full picture, but a gate needs a single number. Two are standard, and they measure different things:

- **ECE** (Expected Calibration Error): for each bin, take the gap `|accuracy - mean_conf|`, then average those gaps weighted by how many predictions fell in each bin. It answers "on average, how far off is the stated confidence from the truth?"
- **Brier score**: the mean squared error between each individual confidence and its individual outcome, `(conf - outcome)^2`, averaged over all predictions. It rewards being right AND appropriately unsure in the same number, without any binning at all.

Both are 0 for a perfect model and both get worse as confidence drifts from reality, but ECE is bin-level while Brier is prediction-level.

In [ ]:
def ece(conf, outcome):
    """Expected Calibration Error: weighted average |accuracy - confidence|."""
    total = len(conf)
    rows = bin_stats(conf, outcome)
    err = 0.0
    for count, mean_conf, acc in rows:
        if count == 0:
            continue
        err += (count / total) * abs(acc - mean_conf)
    return err


def brier(conf, outcome):
    """Brier score: mean over predictions of (confidence - outcome)^2."""
    return sum((c - o) ** 2 for c, o in zip(conf, outcome)) / len(conf)


print(f"ECE (over-confident)   = {ece(CONF, OUTCOME_OVER):.3f}")
print(f"Brier (over-confident) = {brier(CONF, OUTCOME_OVER):.3f}")

## Step 4 — Assembling the full reliability report

The last helpers are bookkeeping: `overall_confidence` and `overall_accuracy` give the two headline numbers, `bin_label` formats a bin's edges for printing, and `report` ties everything together into the reliability curve table plus the two summary metrics, for one model at a time. Running it on the over-confident model first shows the gap building bin by bin: every row's `accuracy` sits well below its `mean_conf`.

In [ ]:
def overall_confidence(conf):
    return sum(conf) / len(conf)


def overall_accuracy(outcome):
    return sum(outcome) / len(outcome)


def bin_label(i):
    lo, hi = BINS[i]
    if lo == hi:
        return f"[{lo:.1f}]     "
    return f"[{lo:.1f}, {hi:.1f})"


def report(conf, outcome, title):
    print(title)
    print(f"  mean confidence = {overall_confidence(conf):.2f}"
          f"   accuracy = {overall_accuracy(outcome):.2f}")
    print("  reliability curve (per confidence bin):")
    print("    bin           n   mean_conf   accuracy   gap")
    rows = bin_stats(conf, outcome)
    for i, (count, mean_conf, acc) in enumerate(rows):
        if count == 0:
            print(f"    {bin_label(i)}   0      -           -         -")
            continue
        gap = abs(acc - mean_conf)
        print(f"    {bin_label(i)}   {count}   "
              f"{mean_conf:8.3f}   {acc:8.3f}   {gap:.3f}")
    e = ece(conf, outcome)
    b = brier(conf, outcome)
    print(f"  ECE   = weighted avg gap  = {e:.3f}")
    print(f"  Brier = mean (conf-out)^2 = {b:.3f}")
    return e, b


e_over, b_over = report(CONF, OUTCOME_OVER,
                        "OVER-CONFIDENT model  (outcomes do NOT track confidence):")

## Step 5 — Same confidences, honest outcomes

Now run the identical `report` on `OUTCOME_CAL`. Nothing about the confidences changes, only whether the outcomes track them. If calibration were really just about the confidences themselves, both runs would score the same. They do not: ECE and Brier both collapse, which is the whole point of measuring calibration separately from accuracy.

In [ ]:
e_cal, b_cal = report(CONF, OUTCOME_CAL,
                      "WELL-CALIBRATED model  (same confidences, outcomes track them):")

print(f"\nECE   {e_over:.3f} -> {e_cal:.3f}   "
      f"(gap between promise and reality shrinks {e_over / e_cal:.1f}x)")
print(f"Brier {b_over:.3f} -> {b_cal:.3f}   "
      "(lower is better; rewards being right AND appropriately unsure)")

## Recap

- **Calibration is separate from accuracy.** Both models here see the exact same 10 confidences (mean 0.82); only whether the outcomes track them differs, and that alone swings ECE from 0.420 to 0.040 and Brier from 0.432 to 0.172.
- **The reliability curve is the by-hand primitive**: bin predictions by stated confidence, compare each bin's mean confidence to its actual accuracy. A well-calibrated model's bins sit near the diagonal; an over-confident model's bins sit below it.
- **ECE and Brier answer different questions.** ECE is bin-level, the weighted average gap between promise and reality. Brier is prediction-level, the mean squared error of confidence against outcome, and rewards being right and appropriately unsure at once.
- **Confidence you cannot trust is worse than no confidence at all.** The over-confident model says 0.82 but is right 40% of the time — every single bin sits below the diagonal, which is a systematic failure, not noise.

Next: regression gates, turning a score plus a confidence interval into a decision rule you would actually stake a deploy on.